# Singly Linked List: Node Chaining and Pointer Traversal

**Session:** 2026-10-30 — OOP with Python  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

While Python lists are dynamic contiguous arrays offering $O(1)$ indexed access, linked data structures offer fundamental insights into object composition, pointer referencing, and memory management. In this notebook, we implement a **Singly Linked List** from first principles using OOP, make it conform to Python's iteration and dunder protocols, and analyze time/space trade-offs.

## Learning Outcomes

- Implement a self-contained `Node` class with data payload and next pointer.
- Manually chain and traverse node sequences.
- Construct a `SinglyLinkedList` container supporting $O(1)$ prepend, $O(n)$ append, lookup, and deletion.
- Implement the `__iter__` and `__len__` dunder protocols to make linked lists idiomatic Python containers.
- Compare algorithmic complexity between linked lists and contiguous arrays (`list`).

## 1. The `Node` Class & Manual Chaining

A node is the atomic building block of pointer-based data structures. It encapsulates:
1. `data`: The payload stored at the node.
2. `next`: A reference to the subsequent `Node` (or `None` if terminal).

In [ ]:
from typing import Any


class Node:
    """A single node in a linked structure."""

    def __init__(self, data: Any = None) -> None:
        self.data: Any = data
        self.next: Node | None = None

    def __repr__(self) -> str:
        return f"Node({self.data!r})"


# Manual node linkage: n1 -> n2 -> n3 -> None
n1 = Node(10)
n2 = Node(20)
n3 = Node(30)

n1.next = n2
n2.next = n3

# Manual traversal
current = n1
values = []
while current is not None:
    values.append(current.data)
    current = current.next

print(f"Manually traversed values: {values}")
assert values == [10, 20, 30]

## 2. Building the `SinglyLinkedList` Container

To manage a chain of nodes cleanly, we wrap the head reference in a `SinglyLinkedList` class supporting standard collection operations:

In [ ]:
class SinglyLinkedList:
    """Singly Linked List container with O(1) prepend and O(n) operations."""

    def __init__(self) -> None:
        self.head: Node | None = None
        self._size: int = 0

    def is_empty(self) -> bool:
        return self.head is None

    def prepend(self, data: Any) -> None:
        """Insert new data at the front of the list (O(1))."""
        new_node = Node(data)
        new_node.next = self.head
        self.head = new_node
        self._size += 1

    def append(self, data: Any) -> None:
        """Insert new data at the end of the list (O(n))."""
        new_node = Node(data)
        if self.head is None:
            self.head = new_node
        else:
            current = self.head
            while current.next is not None:
                current = current.next
            current.next = new_node
        self._size += 1

    def find(self, target: Any) -> Node | None:
        """Search for the first node containing target data (O(n))."""
        current = self.head
        while current is not None:
            if current.data == target:
                return current
            current = current.next
        return None

    def delete(self, target: Any) -> bool:
        """Delete the first node containing target data (O(n))."""
        if self.head is None:
            return False

        # If head is the target
        if self.head.data == target:
            self.head = self.head.next
            self._size -= 1
            return True

        current = self.head
        while current.next is not None and current.next.data != target:
            current = current.next

        if current.next is not None:
            # Unlink the target node
            current.next = current.next.next
            self._size -= 1
            return True

        return False

    def __len__(self) -> int:
        return self._size

    def __iter__(self):
        """Allow seamless iteration using Python's iteration protocol."""
        current = self.head
        while current is not None:
            yield current.data
            current = current.next

    def __repr__(self) -> str:
        nodes = [repr(x) for x in self]
        return " -> ".join(nodes) + " -> None"


# Verify core operations
llist = SinglyLinkedList()
llist.append("Monday")
llist.append("Tuesday")
llist.append("Wednesday")
llist.prepend("Sunday")

print(f"Linked list: {llist}")
print(f"Length: {len(llist)}")
assert len(llist) == 4
assert list(llist) == ["Sunday", "Monday", "Tuesday", "Wednesday"]

# Test deletion
deleted = llist.delete("Tuesday")
assert deleted is True
assert list(llist) == ["Sunday", "Monday", "Wednesday"]
assert len(llist) == 3
print(f"After deletion of 'Tuesday': {llist}")

## 3. Complexity Analysis: Singly Linked List vs Contiguous Array

| Operation | Singly Linked List | Python Dynamic Array (`list`) | Underlying Rationale |
| :--- | :---: | :---: | :--- |
| **Prepend (`insert(0)`)** | $O(1)$ | $O(n)$ | Linked list updates 1 pointer; array shifts all $n$ elements |
| **Append (`insert(end)`)** | $O(n)$ (or $O(1)$ with tail) | $O(1)$ amortized | Array reserves spare capacity; linked list traverses without tail |
| **Index Lookup (`[k]`)** | $O(k)$ | $O(1)$ | Array computes RAM memory address directly via offset calculation |
| **Delete arbitrary key** | $O(n)$ search + $O(1)$ unlink | $O(n)$ search + $O(n)$ shift | Linked list unlinks node; array shifts elements left |
| **Cache Locality** | Poor (scattered heap nodes) | Excellent (contiguous memory buffer) | CPU prefetching benefits dynamic arrays dramatically |

## Summary

- A linked list consists of discrete `Node` objects chained via pointer references.
- Prepending is $O(1)$, avoiding memory shifting costs of dynamic arrays.
- By implementing `__iter__` and `__len__`, custom linked structures integrate seamlessly with Python's standard language features.